In [1]:
import pandas as pd

df = pd.read_excel("screening_results.xlsx")

# Step 1: keep only excluded papers
excluded = df[df["decision"] == "exclude"].copy()

# Step 2: keep only those with valid (non-empty) reasons
def has_valid_reason(x):
    if pd.isna(x):
        return False
    x = str(x).strip().lower()
    return x != "" and "unspecified" not in x and "error" not in x

excluded_valid = excluded[excluded["matched_exclusion_criteria"].apply(has_valid_reason)]

print("Excluded papers (total):", len(excluded))
print("Excluded with clear reasons:", len(excluded_valid))
print("Excluded without clear reasons:", len(excluded) - len(excluded_valid))

Excluded papers (total): 67
Excluded with clear reasons: 66
Excluded without clear reasons: 1


In [3]:
# Get papers WITHOUT clear reasons
excluded_no_reason = excluded[~excluded["matched_exclusion_criteria"].apply(has_valid_reason)]

print("\n--- Papers with NO clear exclusion reason ---\n")

for i, row in excluded_no_reason.iterrows():
    print(f"Title: {row['title']}")
    print(f"Abstract: {row['abstract']}")
    print("-" * 80)


--- Papers with NO clear exclusion reason ---

Title: An audio-visual 3D virtual articulation system for visual speech synthesis,
Abstract: A 3D visible articulation system which utilizes audio-visual data to accurately simulate the 3D articulatory motion is developed in this paper. The 3D articulatory shapes are reconstructed from Magnetic Resonance Images (MRI). In order to synthesize accurate audio-visual results, articulatory parametric models are further developed and the Electromagnetic Articulography (EMA) data with synchronized audio information is utilized to simulate articulation animation. Meanwhile, the collisions between the tongue and other articulators are considered for avoiding physically impossible results. Besides evaluating the results by a widely used distance-based method, which is applied to several key points on the articulatory surface, we develop a shape-based evaluation method to calculate the shape accuracy of the deformable articulator (tongue) in each fra

In [2]:
from collections import Counter

all_reasons = []

for r in excluded_valid["matched_exclusion_criteria"]:
    parts = [x.strip() for x in str(r).split(";") if x.strip()]
    all_reasons.extend(parts)

counts = Counter(all_reasons)

print("\nReason counts:")
for k, v in counts.items():
    print(k, ":", v)

print("\nSum of reason counts:", sum(counts.values()))


Reason counts:
the paper is a review, survey, tutorial, editorial, or non-original study : 7
the abstract does not clearly involve both visual and tactile information : 1
the paper is mainly about haptic rendering : 1
the abstract does not clearly involve both visual and tactile/haptic information : 21
the paper is mainly about teleoperation, VR user study, haptic rendering, or human perception without machine perception : 19
the paper is mainly about tactile sensor development without explicit perception tasks : 1
the paper is mainly about haptic communications and compression rather than perception tasks : 1
the task is not related to perception or recognition : 15
the main focus is grasping, grasp planning, manipulation, robot control, trajectory planning, or pose estimation : 13
the paper is mainly about telepresence : 1
review/survey : 1
the paper is mainly about signal transmission rather than perception : 1
the paper is mainly about haptic rendering or human perception without 

In [4]:
import pandas as pd

df = pd.read_excel("screening_results.xlsx")

# keep only excluded
excluded = df[df["decision"] == "exclude"].copy()

# count number of reasons per paper
def count_reasons(x):
    if pd.isna(x):
        return 0
    return len([r for r in str(x).split(";") if r.strip()])

excluded["num_reasons"] = excluded["matched_exclusion_criteria"].apply(count_reasons)

# papers with multiple reasons
multi_reason = excluded[excluded["num_reasons"] > 1]

print("Number of papers with multiple exclusion reasons:", len(multi_reason))

# show them
for i, row in multi_reason.iterrows():
    print("\n---")
    print("Title:", row["title"])
    print("Reasons:", row["matched_exclusion_criteria"])

Number of papers with multiple exclusion reasons: 20

---
Title: Optum: A Three-in-One Multimodal Tactile Sensor Based on Optical Fiber Knots for On-Orbit Service,
Reasons: the paper is mainly about teleoperation, VR user study, haptic rendering, or human perception without machine perception; the abstract does not clearly involve both visual and tactile/haptic information

---
Title: GelBelt: A Vision-Based Tactile Sensor for Continuous Sensing of Large Surfaces,
Reasons: the paper is mainly about teleoperation, VR user study, haptic rendering, or human perception without machine perception; the task is not related to perception or recognition

---
Title: A novel multisensory device for the assessment and rehabilitation of perceptual and attentional competencies,
Reasons: the main focus is grasping, grasp planning, manipulation, robot control, trajectory planning, or pose estimation; the paper is mainly about teleoperation, VR user study, haptic rendering, or human perception without 